Before EGU 2026, I decided to re-make the passage of OPT operator derivation since we need to take care of staggered grids etc. which are not easy to handle with the current version

# Big change!!

pointsInSpace, pointsInTime should be real (before they were -1 but now it should be 2,3,4 ...)

In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")

include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

using .commonBatchs, .planet1D, .GeoPoints

  Activating project at `~/Documents/Github/flexOPT`


devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]
→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


In [2]:
# Since this is the developing version, I do not use the module from flexOPT.jl

In [3]:

    include("../src/batchFiles/batchSymbolics.jl")
    export x,y,z,t
    export ∂x,∂y,∂z,∂t
    export ∂x²,∂y²,∂z²,∂t² 
    export Num2Float64,usefulPartials,myCoeff,mySolvefor,mySimplify,integrateTaylorPolynomials



use some solid packages

In [26]:
include("../src/motorsOPT/others.jl")

BouncingCoordinates (generic function with 1 method)

In [5]:
    include("../src/motorsOPT/famousEquations.jl")

famousEquation (generic function with 15 methods)

input parameters

In [6]:
famousEquationType="2DacousticTime"
Δnum = (1.0,1.0,1.0)

orderBtime=-1
orderBspace=-1
pointsInSpace=3
pointsInTime=3

# new parameters for interpolated Taylor expansion
# μ points should be distributed from y_min+offsetμFromExtremitiesInΔy*Δy to y_max-offsetμFromExtremitiesInΔy*Δy
pointsμInSpace = 2
pointsμInTime = 2
offsetμFromExtremitiesInΔyInSpace=1/2
offsetμFromExtremitiesInΔyInTime=1/2

WorderBspace=-1
WorderBtime=-1
supplementaryOrder=2

2

We also propose different ways of boundary treatment (ghosting: Neumann/Dirichlet; same number of points etc.)

Starting the OPT derivation

In [11]:
concreteParametersForOPTConstruction = @strdict famousEquationType Δnum orderBtime orderBspace WorderBtime WorderBspace supplementaryOrder pointsInSpace pointsInTime 

# construction of NamedTuples
trialFunctionsCharacteristics=(orderBtime=orderBtime,orderBspace=orderBspace,pointsInSpace=pointsInSpace,pointsInTime=pointsInTime)
TaylorOptions=(WorderBtime=WorderBtime,WorderBspace=WorderBspace,supplementaryOrder=supplementaryOrder,pointsμInSpace=pointsμInSpace,pointsμInTime=pointsμInTime,offsetμInΔyInSpace=offsetμFromExtremitiesInΔyInSpace,offsetμInΔyInTime=offsetμFromExtremitiesInΔyInTime)


(WorderBtime = -1, WorderBspace = -1, supplementaryOrder = 2, pointsμInSpace = 2, pointsμInTime = 2, offsetμInΔyInSpace = 0.5, offsetμInΔyInTime = 0.5)

In [27]:
equationCharacteristics=famousEquations(famousEquationType)
numbersOfTheSystem=numbersOfTheExpression(equationCharacteristics,trialFunctionsCharacteristics,TaylorOptions)
dependencies,ordersForSplines,configsTaylor=investigateDependencies(equationCharacteristics,numbersOfTheSystem,trialFunctionsCharacteristics,TaylorOptions)
bigα,varM=bigαFinder(equationCharacteristics,numbersOfTheSystem,ordersForSplines)
@show bigα,varM
@show configsTaylor


(bigα, varM) = (Any[Any[(node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(3, 1, 1)), (node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 3, 1)), (node = 1, nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 1, 3))];;], Any[v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉ v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉ v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉])
configsTaylor = (multiOrdersIndices = CartesianIndices((5, 5, 5)), availablePointsConfigurations = [[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]], centrePointConfigurations = [14])


(multiOrdersIndices = CartesianIndices((5, 5, 5)), availablePointsConfigurations = [[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]], centrePointConfigurations = [14])

In [34]:
configsTaylor.availablePointsConfigurations[1][2]

3-element Vector{Int64}:
 2
 1
 1

In [29]:
configsTaylor.availablePointsConfigurations[1][configsTaylor.centrePointConfigurations[1]]

3-element Vector{Int64}:
 2
 2
 2

In [31]:
@show numbersOfTheSystem.pointsμUsed
@show numbersOfTheSystem.offsetsμUsed

numbersOfTheSystem.pointsμUsed = [2, 2, 2]
numbersOfTheSystem.offsetsμUsed = [0.5, 0.5, 0.5]


3-element Vector{Float64}:
 0.5
 0.5
 0.5